# Mapa de pixels dos monitores

Aplicativo Windows em uma única célula, usando apenas a biblioteca padrão do Python. Execute a célula abaixo. A janela de controle e o overlay serão abertos fora do navegador.

O PDF é produzido por um escritor vetorial incorporado ao próprio código. Nenhum pacote precisa ser instalado com `pip`. O ambiente Python precisa incluir Tkinter, como normalmente ocorre na instalação oficial do Python para Windows.


In [ ]:
# Mapa de pixels dos monitores - aplicativo em uma unica celula
# Requisitos: Windows 10/11, Python 3 e Tkinter (biblioteca padrao do Python).
# Nao requer pip install.

import ctypes
from ctypes import wintypes
import math
import os
import tkinter as tk
from tkinter import ttk, filedialog, messagebox

if os.name != "nt":
    raise RuntimeError("Este notebook-app foi projetado para Windows 10/11.")

# Coordenadas fisicas, inclusive com escalas diferentes entre monitores.
try:
    ctypes.windll.user32.SetProcessDpiAwarenessContext(ctypes.c_void_p(-4))
except Exception:
    try:
        ctypes.windll.shcore.SetProcessDpiAwareness(2)
    except Exception:
        pass

user32 = ctypes.windll.user32
GWL_EXSTYLE = -20
WS_EX_LAYERED = 0x00080000
WS_EX_TRANSPARENT = 0x00000020
WS_EX_NOACTIVATE = 0x08000000
MONITORINFOF_PRIMARY = 1

class RECT(ctypes.Structure):
    _fields_ = [("left", wintypes.LONG), ("top", wintypes.LONG),
                ("right", wintypes.LONG), ("bottom", wintypes.LONG)]

class MONITORINFOEXW(ctypes.Structure):
    _fields_ = [("cbSize", wintypes.DWORD), ("rcMonitor", RECT),
                ("rcWork", RECT), ("dwFlags", wintypes.DWORD),
                ("szDevice", wintypes.WCHAR * 32)]

def listar_monitores():
    monitores = []
    proc_type = ctypes.WINFUNCTYPE(wintypes.BOOL, wintypes.HMONITOR,
                                  wintypes.HDC, ctypes.POINTER(RECT), wintypes.LPARAM)
    def callback(hmon, hdc, lprect, lparam):
        info = MONITORINFOEXW()
        info.cbSize = ctypes.sizeof(info)
        user32.GetMonitorInfoW(hmon, ctypes.byref(info))
        r = info.rcMonitor
        monitores.append({
            "handle": int(hmon), "nome": info.szDevice,
            "x": r.left, "y": r.top,
            "largura": r.right - r.left, "altura": r.bottom - r.top,
            "principal": bool(info.dwFlags & MONITORINFOF_PRIMARY)
        })
        return True
    cb = proc_type(callback)
    user32.EnumDisplayMonitors(0, 0, cb, 0)
    monitores.sort(key=lambda m: (not m["principal"], m["x"], m["y"]))
    return monitores

def pdf_escape(texto):
    return texto.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")

def passo_bom(extensao, alvo=12):
    bruto = max(1.0, extensao / alvo)
    base = 10 ** math.floor(math.log10(bruto))
    for fator in (1, 2, 5, 10):
        if fator * base >= bruto:
            return int(fator * base)
    return int(10 * base)

def gerar_pdf_vetorial(caminho, rect, monitor, orientacao):
    # Escritor PDF minimo, vetorial e inteiramente baseado na biblioteca padrao.
    # Pagina A4 em pontos: 595 x 842 (portrait) ou 842 x 595 (landscape).
    x0, y0, largura_px, altura_px = rect
    if orientacao == "Automatica":
        orientacao = "Landscape" if largura_px >= altura_px else "Portrait"
    pw, ph = ((842.0, 595.0) if orientacao == "Landscape" else (595.0, 842.0))
    margem, topo = 42.0, 82.0
    area_w, area_h = pw - 2*margem, ph - topo - margem
    escala = min(area_w/largura_px, area_h/altura_px)
    dw, dh = largura_px*escala, altura_px*escala
    px0 = margem + (area_w-dw)/2
    py0 = margem + (area_h-dh)/2
    comandos = []
    comandos.append("1 1 1 rg 0 0 %.2f %.2f re f" % (pw, ph))
    comandos.append("0.08 0.08 0.08 rg BT /F1 15 Tf %.2f %.2f Td (%s) Tj ET" %
                    (margem, ph-34, pdf_escape("Mapa de pixels da selecao")))
    origem_global_x = monitor["x"] + x0
    origem_global_y = monitor["y"] + y0
    subtitulo = (f"Monitor {monitor['nome']} | selecao ({x0}, {y0}) | "
                 f"{largura_px} x {altura_px} px | origem global "
                 f"({origem_global_x}, {origem_global_y})")
    comandos.append("0.18 0.18 0.18 rg BT /F1 8 Tf %.2f %.2f Td (%s) Tj ET" %
                    (margem, ph-54, pdf_escape(subtitulo)))
    comandos.append("0.98 0.99 1 rg %.2f %.2f %.2f %.2f re f" % (px0, py0, dw, dh))
    sx, sy = passo_bom(largura_px), passo_bom(altura_px)
    comandos.append("0.55 0.70 0.82 RG 0.45 w")
    x = 0
    while x <= largura_px:
        xx = px0 + x*escala
        comandos.append("%.2f %.2f m %.2f %.2f l S" % (xx, py0, xx, py0+dh))
        comandos.append("0.08 0.28 0.45 rg BT /F1 6 Tf %.2f %.2f Td (%d) Tj ET" %
                        (xx+1.5, py0+dh-9, x0+x))
        x += sx
    y = 0
    while y <= altura_px:
        yy = py0 + dh - y*escala
        comandos.append("0.55 0.70 0.82 RG %.2f %.2f m %.2f %.2f l S" % (px0, yy, px0+dw, yy))
        comandos.append("0.08 0.28 0.45 rg BT /F1 6 Tf %.2f %.2f Td (%d) Tj ET" %
                        (px0+2, yy-7, y0+y))
        y += sy
    comandos.append("0 0.25 0.48 RG 1.1 w %.2f %.2f %.2f %.2f re S" % (px0, py0, dw, dh))
    stream = ("\n".join(comandos) + "\n").encode("latin-1", "replace")
    objetos = [
        b"<< /Type /Catalog /Pages 2 0 R >>",
        b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
        ("<< /Type /Page /Parent 2 0 R /MediaBox [0 0 %.2f %.2f] "
         "/Resources << /Font << /F1 5 0 R >> >> /Contents 4 0 R >>" % (pw, ph)).encode(),
        b"<< /Length " + str(len(stream)).encode() + b" >>\nstream\n" + stream + b"endstream",
        b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>"
    ]
    dados = bytearray(b"%PDF-1.4\n%\xe2\xe3\xcf\xd3\n")
    offsets = [0]
    for i, obj in enumerate(objetos, 1):
        offsets.append(len(dados))
        dados.extend(f"{i} 0 obj\n".encode() + obj + b"\nendobj\n")
    xref = len(dados)
    dados.extend(f"xref\n0 {len(objetos)+1}\n".encode())
    dados.extend(b"0000000000 65535 f \n")
    for off in offsets[1:]:
        dados.extend(f"{off:010d} 00000 n \n".encode())
    dados.extend((f"trailer\n<< /Size {len(objetos)+1} /Root 1 0 R >>\n"
                  f"startxref\n{xref}\n%%EOF\n").encode())
    with open(caminho, "wb") as arq:
        arq.write(dados)

class AppMapaPixels:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Mapa de pixels dos monitores")
        self.root.geometry("720x270")
        self.root.protocol("WM_DELETE_WINDOW", self.fechar)
        self.monitores = listar_monitores()
        if not self.monitores:
            raise RuntimeError("Nenhum monitor foi detectado.")
        self.overlay = None
        self.canvas = None
        self.indice = tk.IntVar(value=0)
        self.opacidade = tk.IntVar(value=70)
        self.orientacao = tk.StringVar(value="Automatica")
        self.selecao = None
        self.ponto1 = None
        self.modo_selecao = False
        self.hint_job = None
        self.montar_painel()
        self.criar_overlay()

    def montar_painel(self):
        quadro = ttk.Frame(self.root, padding=12)
        quadro.pack(fill="both", expand=True)
        ttk.Label(quadro, text="Monitor:").grid(row=0, column=0, sticky="w", pady=4)
        nomes = []
        for i, m in enumerate(self.monitores, 1):
            sufixo = " principal" if m["principal"] else ""
            nomes.append(f"Monitor {i}: {m['nome']} | {m['largura']} x {m['altura']} | origem {m['x']}, {m['y']}{sufixo}")
        self.combo = ttk.Combobox(quadro, values=nomes, state="readonly", width=78)
        self.combo.current(0)
        self.combo.grid(row=0, column=1, columnspan=4, sticky="ew", pady=4)
        self.combo.bind("<<ComboboxSelected>>", lambda e: self.trocar_monitor())
        ttk.Label(quadro, text="Opacidade:").grid(row=1, column=0, sticky="w", pady=4)
        escala = ttk.Scale(quadro, from_=10, to=100, variable=self.opacidade,
                           command=lambda v: self.redesenhar())
        escala.grid(row=1, column=1, sticky="ew")
        ttk.Label(quadro, textvariable=self.opacidade, width=4).grid(row=1, column=2, sticky="w")
        ttk.Label(quadro, text="Orientacao PDF:").grid(row=1, column=3, sticky="e")
        ttk.Combobox(quadro, textvariable=self.orientacao,
                     values=("Automatica", "Portrait", "Landscape"),
                     state="readonly", width=13).grid(row=1, column=4, sticky="w")
        ttk.Button(quadro, text="Mostrar/ocultar mapa", command=self.alternar).grid(row=2, column=0, pady=10, sticky="ew")
        ttk.Button(quadro, text="Selecionar por 2 pontos", command=self.iniciar_selecao).grid(row=2, column=1, pady=10, sticky="ew")
        ttk.Button(quadro, text="Usar tela inteira", command=self.tela_inteira).grid(row=2, column=2, pady=10, sticky="ew")
        self.bt_pdf = ttk.Button(quadro, text="Gerar PDF", command=self.exportar_pdf, state="disabled")
        self.bt_pdf.grid(row=2, column=3, columnspan=2, pady=10, sticky="ew")
        self.status = tk.StringVar(value="Overlay ativo e permeavel a cliques.")
        ttk.Label(quadro, textvariable=self.status, wraplength=680).grid(row=3, column=0, columnspan=5, sticky="w", pady=6)
        ttk.Label(quadro, text="Durante a selecao, o overlay captura somente os dois cliques; depois volta ao modo click-through.",
                  foreground="#555555", wraplength=680).grid(row=4, column=0, columnspan=5, sticky="w")
        quadro.columnconfigure(1, weight=1)

    def monitor(self):
        return self.monitores[self.combo.current()]

    def criar_overlay(self):
        if self.overlay:
            self.overlay.destroy()
        m = self.monitor()
        self.overlay = tk.Toplevel(self.root)
        self.overlay.overrideredirect(True)
        self.overlay.attributes("-topmost", True)
        self.overlay.configure(bg="#ff00ff")
        self.overlay.wm_attributes("-transparentcolor", "#ff00ff")
        geo = f"{m['largura']}x{m['altura']}{m['x']:+d}{m['y']:+d}"
        self.overlay.geometry(geo)
        self.canvas = tk.Canvas(self.overlay, bg="#ff00ff", highlightthickness=0,
                                width=m["largura"], height=m["altura"])
        self.canvas.pack(fill="both", expand=True)
        self.overlay.update_idletasks()
        self.definir_clickthrough(True)
        self.redesenhar()
        self.atualizar_hint()

    def definir_clickthrough(self, ativo):
        hwnd = self.overlay.winfo_id()
        estilo = user32.GetWindowLongW(hwnd, GWL_EXSTYLE)
        estilo |= WS_EX_LAYERED | WS_EX_NOACTIVATE
        estilo = (estilo | WS_EX_TRANSPARENT) if ativo else (estilo & ~WS_EX_TRANSPARENT)
        user32.SetWindowLongW(hwnd, GWL_EXSTYLE, estilo)

    def cor(self, rgb, fator=1.0):
        # Tk nao oferece alfa por item; aproximacao por mistura com fundo visual.
        r, g, b = rgb
        intensidade = max(0.1, min(1.0, self.opacidade.get()/100 * fator))
        r = int(255-(255-r)*intensidade)
        g = int(255-(255-g)*intensidade)
        b = int(255-(255-b)*intensidade)
        return f"#{r:02x}{g:02x}{b:02x}"

    def redesenhar(self):
        if not self.canvas:
            return
        self.canvas.delete("grade")
        m = self.monitor()
        for x in range(0, m["largura"]+1, 10):
            principal = x % 100 == 0
            cor = self.cor((0, 105, 185), 1.0) if principal else self.cor((40, 155, 215), 0.45)
            if principal:
                self.canvas.create_line(x, 0, x, m["altura"], fill=cor, tags="grade")
                if x:
                    self.canvas.create_text(x+3, 4, text=str(x), fill=cor, anchor="nw", font=("Segoe UI", 8), tags="grade")
            else:
                t = 8 if x % 50 == 0 else 4
                self.canvas.create_line(x, 0, x, t, fill=cor, tags="grade")
        for y in range(0, m["altura"]+1, 10):
            principal = y % 100 == 0
            cor = self.cor((0, 105, 185), 1.0) if principal else self.cor((40, 155, 215), 0.45)
            if principal:
                self.canvas.create_line(0, y, m["largura"], y, fill=cor, tags="grade")
                if y:
                    self.canvas.create_text(3, y-3, text=str(y), fill=cor, anchor="sw", font=("Segoe UI", 8), tags="grade")
            else:
                t = 8 if y % 50 == 0 else 4
                self.canvas.create_line(0, y, t, y, fill=cor, tags="grade")
        self.desenhar_selecao()

    def atualizar_hint(self):
        if not self.overlay or not self.overlay.winfo_exists():
            return
        self.canvas.delete("hint")
        m = self.monitor()
        gx, gy = self.root.winfo_pointerx(), self.root.winfo_pointery()
        x, y = gx-m["x"], gy-m["y"]
        if 0 <= x < m["largura"] and 0 <= y < m["altura"]:
            texto = f"x={x} px  y={y} px | global {gx}, {gy}"
            hx, hy = x+15, y+22
            if hx+235 > m["largura"]: hx = max(2, x-245)
            if hy+25 > m["altura"]: hy = max(2, y-30)
            self.canvas.create_rectangle(hx, hy, hx+235, hy+23, fill="#202020", outline="#202020", tags="hint")
            self.canvas.create_text(hx+6, hy+11, text=texto, fill="white", anchor="w",
                                    font=("Segoe UI", 8), tags="hint")
        self.canvas.tag_raise("hint")
        self.hint_job = self.root.after(35, self.atualizar_hint)

    def trocar_monitor(self):
        self.selecao = None
        self.bt_pdf.configure(state="disabled")
        self.criar_overlay()
        self.status.set("Monitor alterado. Overlay ativo e permeavel a cliques.")

    def alternar(self):
        if self.overlay.state() == "withdrawn":
            self.overlay.deiconify(); self.overlay.attributes("-topmost", True)
            self.status.set("Overlay visivel.")
        else:
            self.overlay.withdraw(); self.status.set("Overlay oculto.")

    def iniciar_selecao(self):
        self.overlay.deiconify()
        self.modo_selecao = True
        self.ponto1 = None
        self.selecao = None
        self.bt_pdf.configure(state="disabled")
        self.definir_clickthrough(False)
        self.canvas.configure(cursor="crosshair")
        self.canvas.bind("<Button-1>", self.clique_selecao)
        self.root.bind("<Escape>", self.cancelar_selecao)
        self.status.set("Clique no primeiro canto e depois no canto oposto. Esc cancela.")

    def clique_selecao(self, evento):
        if self.ponto1 is None:
            self.ponto1 = (evento.x, evento.y)
            self.status.set(f"Primeiro ponto: ({evento.x}, {evento.y}). Indique o canto oposto.")
        else:
            x1, y1 = self.ponto1
            x2, y2 = evento.x, evento.y
            x, y = min(x1,x2), min(y1,y2)
            w, h = abs(x2-x1), abs(y2-y1)
            if w < 2 or h < 2:
                self.status.set("Selecao muito pequena. Repita a operacao.")
                self.ponto1 = None
                return
            self.selecao = (x, y, w, h)
            self.encerrar_modo_selecao()
            self.bt_pdf.configure(state="normal")
            self.status.set(f"Area: x={x}, y={y}, {w} x {h} px.")
            self.desenhar_selecao()

    def encerrar_modo_selecao(self):
        self.modo_selecao = False
        self.canvas.unbind("<Button-1>")
        self.root.unbind("<Escape>")
        self.canvas.configure(cursor="")
        self.definir_clickthrough(True)

    def cancelar_selecao(self, evento=None):
        self.ponto1 = None
        self.selecao = None
        self.encerrar_modo_selecao()
        self.canvas.delete("selecao")
        self.bt_pdf.configure(state="disabled")
        self.status.set("Selecao cancelada. Overlay novamente permeavel a cliques.")

    def desenhar_selecao(self):
        if not self.canvas:
            return
        self.canvas.delete("selecao")
        if self.selecao:
            x, y, w, h = self.selecao
            self.canvas.create_rectangle(x, y, x+w, y+h, outline="#d54b00", width=2,
                                         dash=(7,4), tags="selecao")
            self.canvas.create_text(x+5, y+5, text=f"{w} x {h} px", fill="#7a2500",
                                    anchor="nw", font=("Segoe UI", 9, "bold"), tags="selecao")

    def tela_inteira(self):
        m = self.monitor()
        self.selecao = (0, 0, m["largura"], m["altura"])
        self.bt_pdf.configure(state="normal")
        self.status.set(f"Tela inteira: {m['largura']} x {m['altura']} px.")
        self.desenhar_selecao()

    def exportar_pdf(self):
        if not self.selecao:
            return
        x, y, w, h = self.selecao
        caminho = filedialog.asksaveasfilename(title="Salvar mapa em PDF",
                    defaultextension=".pdf", filetypes=[("PDF", "*.pdf")],
                    initialfile=f"mapa_pixels_{w}x{h}.pdf")
        if not caminho:
            return
        try:
            gerar_pdf_vetorial(caminho, self.selecao, self.monitor(), self.orientacao.get())
            if os.path.getsize(caminho) <= 0:
                raise IOError("arquivo vazio")
            self.status.set(f"PDF gerado: {caminho}")
            messagebox.showinfo("PDF gerado", f"Arquivo salvo em:\n{caminho}")
        except Exception as exc:
            messagebox.showerror("Falha ao gerar PDF", str(exc))

    def fechar(self):
        if self.hint_job:
            try: self.root.after_cancel(self.hint_job)
            except Exception: pass
        if self.overlay:
            self.overlay.destroy()
        self.root.destroy()

    def executar(self):
        self.root.mainloop()

# Feche uma instancia anterior ao executar novamente a celula, se existir.
try:
    app_mapa_pixels.fechar()
except Exception:
    pass

app_mapa_pixels = AppMapaPixels()
app_mapa_pixels.executar()
